# Chapter 17: Parsing Structured Data with Python

Companion notebook to **python-from-zero** · `lesson-17` · based on *Python for VLSI*, Chapter 17.

### You will learn

- Reading lines, stripping whitespace, splitting on delimiters
- Filtering and extracting values with string methods
- Parsing with `re.search` for patterns that aren't simple splits
- Bringing it together: pull every slack from an STA report and rank the worst

In [1]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()}")
assert sys.version_info >= (3, 9), "This notebook needs Python 3.9 or newer."

Python 3.12.3 on Linux


## Setup — write a small STA report we can parse

In [2]:
from pathlib import Path
Path('sta.rpt').write_text(
    'Startpoint: U1/clk\n'
    'Endpoint: U2/data\n'
    'Path Group: clk\n'
    'slack -0.045 setup violation\n'
    'slack 0.120 hold met\n'
    'slack -0.089 setup violation\n'
)

132

## Read and split lines into tokens

In [3]:
with open('sta.rpt') as f:
    for line in f:
        tokens = line.split()
        print(tokens)

['Startpoint:', 'U1/clk']
['Endpoint:', 'U2/data']
['Path', 'Group:', 'clk']
['slack', '-0.045', 'setup', 'violation']
['slack', '0.120', 'hold', 'met']
['slack', '-0.089', 'setup', 'violation']


## Extract slack values

In [4]:
with open('sta.rpt') as f:
    for line in f:
        if line.startswith('slack'):
            tokens = line.split()
            slack = float(tokens[1])
            print('Slack value:', slack)

Slack value: -0.045
Slack value: 0.12
Slack value: -0.089


## Delimited files — CSV

In [5]:
from pathlib import Path
Path('nets.csv').write_text(
    'clk,0.2,high_fanout\n'
    'reset,0.1,low_fanout\n'
    'data,0.25,high_fanout\n'
)

with open('nets.csv') as f:
    for line in f:
        tokens = line.strip().split(',')
        net_name = tokens[0]
        fanout = float(tokens[1])
        print(net_name, fanout)

clk 0.2
reset 0.1
data 0.25


## Filter lines

In [6]:
with open('sta.rpt') as f:
    for line in f:
        if line.startswith('slack') and 'violation' in line:
            print('Violation:', line.strip())

Violation: slack -0.045 setup violation
Violation: slack -0.089 setup violation


## Store, sort, and report the worst

In [7]:
violations = []
with open('sta.rpt') as f:
    for line in f:
        if line.startswith('slack'):
            tokens = line.split()
            try:
                slack = float(tokens[1])
                violations.append(slack)
            except ValueError:
                continue
violations.sort()
print('worst slack:', violations[0])

worst slack: -0.089


## Regular expressions for patterns that aren't simple splits

In [8]:
from pathlib import Path
Path('sta_regex.rpt').write_text(
    'slack: -0.045 (VIOLATION)\n'
    'slack: 0.110 (OK)\n'
)
import re
with open('sta_regex.rpt') as f:
    for line in f:
        match = re.search(r'slack:\s*(-?\d+\.\d+)', line)
        if match:
            print('Slack:', float(match.group(1)))

Slack: -0.045
Slack: 0.11


## VLSI use case — extract net names

In [9]:
from pathlib import Path
Path('nets_report.txt').write_text('Net: clk\nNet: reset\nNet: data\n')

nets = []
with open('nets_report.txt') as f:
    for line in f:
        if 'Net:' in line:
            tokens = line.split()
            idx = tokens.index('Net:')
            nets.append(tokens[idx + 1])
print(nets)

['clk', 'reset', 'data']


## Pitfalls to watch for

- Forgetting `.strip()` — trailing `\n` ends up in tokens.
- `IndexError` when a line has fewer tokens than expected — guard with `len(tokens)`.
- Comparing strings to numbers — remember to cast with `float(...)`.

In [10]:
line = 'slack -0.03 violation\n'
tokens = line.split()
if len(tokens) >= 2:
    slack = float(tokens[1])
    print('slack numeric:', slack, 'is negative?', slack < 0)

slack numeric: -0.03 is negative? True


## Exercise 1

From `sta.rpt`, print only `violation` lines using `startswith('slack')` and `in`.

In [11]:
# Your code here


<details><summary>Show solution</summary>

```python
with open('sta.rpt') as f:
    for line in f:
        if line.startswith('slack') and 'violation' in line:
            print(line.strip())
```

</details>

## Exercise 2

Parse `nets.csv` into a list of (name, fanout) tuples and print those with fanout ≥ 0.2.

In [12]:
# Your code here


<details><summary>Show solution</summary>

```python
rows = []
with open('nets.csv') as f:
    for line in f:
        name, fanout, _ = line.strip().split(',')
        rows.append((name, float(fanout)))
for name, fan in rows:
    if fan >= 0.2:
        print(name, fan)
```

</details>

## Exercise 3

Using `re`, extract every float from `sta_regex.rpt` and print them sorted ascending.

In [13]:
# Your code here


<details><summary>Show solution</summary>

```python
import re
vals = []
with open('sta_regex.rpt') as f:
    for line in f:
        m = re.search(r'(-?\d+\.\d+)', line)
        if m:
            vals.append(float(m.group(1)))
vals.sort()
print(vals)
```

</details>

## Recap

- Read lines, strip whitespace, split on the right delimiter.
- Filter with `startswith`, `in`, or a combination.
- Cast tokens to `float`/`int` before comparing as numbers.
- Reach for `re.search` when the pattern isn't a clean split.

## What's next

You've reached the end of the companion series. Return to the [python-from-zero](https://python-from-zero.pages.dev) site to keep going.